In this notebook, we will test an agent that queries a SQL database using natural language by translating the user’s question into SQL. For a given question related to the data in a SQL database, the results of the agent’s generated queries will be compared with those of manually written queries.

In [1]:
%pip install jsonschema
%pip install langchain
%pip install langchain-ollama
%pip install pandas
%pip install sqlalchemy
%pip install sqlalchemy
%pip install langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


First, we create the agent. For more details, see the sample notebook `single_sql.ipynb`.

In [2]:
import sqlite3
import os
from langchain.tools import tool
import sqlite3
from sqlalchemy import ForeignKey
from sqlalchemy import String, Integer, Boolean, Date
from sqlalchemy.orm import DeclarativeBase
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship
from sqlalchemy import create_engine, MetaData, Table, Column
from sqlalchemy import inspect
from langchain_community.utilities.sql_database import SQLDatabase
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

sqlite_db="./data/employee.db"
sqlite_db_create="./data/employee.sql"
def init_sqlite3_db():
    try:
        if not os.path.exists(sqlite_db):
            connection = sqlite3.connect(sqlite_db)
            with open(sqlite_db_create, "r") as infile:
              content = infile.read() # Reads entire remaining content as a string
            #print(content)
            connection.executescript(content)
            connection.commit()
            connection.close()
        
    except Exception as e:
        print(f"Error running SQL query: {e}")

def query_sqlite3_db(query: str) -> list[any]:
    matches=None;
    try:
        conn = sqlite3.connect(sqlite_db)
        c = conn.cursor()
        c.execute(query)
        matches = c.fetchall()
    except Exception as e:
        print(f"Error running SQL query: {e}")
        matches=[f"Error running SQL query: {e}"]
    finally:
        c.close()
        conn.close()
    return matches
    
init_sqlite3_db();


@tool
def query_sql(query: str) -> list[any]:
    """Run a SQL query on the provided data."""
    try:
        return  query_sqlite3_db(query)
    except Exception as e:  
        serr=f"Error running SQL query: {e}"
        return [serr]  


engine = create_engine("sqlite:///data/employee.db", echo=False)
inspector = inspect(engine)
table_names=inspector.get_table_names()
db = SQLDatabase(engine)
sql_schema=db.get_table_info_no_throw([t.strip() for t in table_names])
from langchain.agents import create_agent
from langchain_mistralai.chat_models import ChatMistralAI
import os
import io

template = f"""
Create a SQL query to get the required information from the data.

Here is the schema for the SQL database you will work with:
{sql_schema}

Given an input question, create a syntactically correct SQLite query to run, then look at the results of the query and return the answer.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.
Only use tools if you are sure a query will get a correct answer. 

tools:
query_sql: returns the entities using a given SQL query.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
Always use double quotes for column names.

Always use the available tools to fetch data.
Only use the results from tools defined above to answer the question.
DO NOT use sample data shown in the table definitions!
DO NOT restrict the number of results with LIMIT clause!
"""


llm =  ChatOllama(
    model="qwen3.6:35b-a3b",#gemma4:31b-cloud",#"qwen3-coder-next:cloud",#"kimi-k2.7-code:cloud",#"qwen3.5:397b-cloud",#"gpt-oss:120b-cloud",#"qwen3.6:35b-a3b",#"qwen3.5:cloud",#qwen3.6:35b-a3b",#"gpt-oss:20b",
    temperature=0,
    max_retries=2,
    base_url="http://192.168.0.124:11434",
    think=True,
    num_ctx=8192,
)


agent = create_agent(
    model=llm,
    tools=[query_sql],
    system_prompt=template,
)

c:\Python313\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


The result of the agent invocation is the full agent history. The last message in this history is the final answer provided by the agent.

The message immediately before the final answer is a `ToolMessage`, which contains the data fetched by the query from the SQL database. We use this message to compare it with the results of the corresponding manually written SQL query as a reference.

For this purpose, we create a convenience function called `get_answer`. The return value of this function includes the parameter `"answer"`, which contains the query result as JSON string. The `"isOk"` property of the return value indicates whether the tool call was successful.

In [3]:
from langchain.messages import ToolMessage 
from pprint import pprint

def checkAgentCall(output):
    if len(output['messages'])>=4 and type(output['messages'][-2]) is ToolMessage:
        return{
            "firstTry":len(output['messages'])==4,
            "isOk": type(output['messages'][-2]) is ToolMessage,
            "answer": output['messages'][-2].content,
            "query" : output['messages'][-3].tool_calls[0]['args']['query']
        }
    else:
        return {
         "firstTry": True,
         "isOk": False,
         "answer": "",
         "query" : ""
       }
    
def get_answer(question):
    output=agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
    )
    pprint(output)
    ana=checkAgentCall(output)
    
    value=output["messages"][-1]
    if value != None and hasattr(value, 'content') and value.content is not None:
        if type(value.content) == str:
            ana['modelAnswer']=value.content.strip()
            return ana
        else:
            ana['modelAnswer']=f"Error:{str(value.content)}"
            return ana
    else:
        ana['modelAnswer']=f"No data!"
        return ana

We can now test whether our function works.

In [4]:
get_answer("Please give me the count of employee names.")

{'messages': [HumanMessage(content='Please give me the count of employee names.', additional_kwargs={}, response_metadata={}, id='9c7bff4b-1ed6-4a9e-94c0-b914191ffad6'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b-a3b', 'created_at': '2026-06-30T09:48:47.250972212Z', 'done': True, 'done_reason': 'stop', 'total_duration': 97870147232, 'load_duration': 43977968550, 'prompt_eval_count': 4915, 'prompt_eval_duration': 43492912000, 'eval_count': 205, 'eval_duration': 10385377000, 'logprobs': None, 'model_name': 'qwen3.6:35b-a3b', 'model_provider': 'ollama'}, id='lc_run--019f17ec-e689-7b42-8693-a48bb494003d-0', tool_calls=[{'name': 'query_sql', 'args': {'query': 'SELECT COUNT(name) FROM employees'}, 'id': '1d2c6d96-259f-48cc-951a-d52f989da6ba', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 4915, 'output_tokens': 205, 'total_tokens': 5120}),
              ToolMessage(content='[[201]]', name='query_sql', id='

{'firstTry': True,
 'isOk': True,
 'answer': '[[201]]',
 'query': 'SELECT COUNT(name) FROM employees',
 'modelAnswer': 'There are 201 employee names in the database.'}

In [5]:
get_answer("Please give me the tasks descriptions of employee Tommy Parker.")

{'messages': [HumanMessage(content='Please give me the tasks descriptions of employee Tommy Parker.', additional_kwargs={}, response_metadata={}, id='767f3a34-a70f-49af-bad3-2ddf8880b795'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b-a3b', 'created_at': '2026-06-15T17:59:49.826245084Z', 'done': True, 'done_reason': 'stop', 'total_duration': 56368662489, 'load_duration': 223838286, 'prompt_eval_count': 1026, 'prompt_eval_duration': 9850403000, 'eval_count': 901, 'eval_duration': 46282842000, 'logprobs': None, 'model_name': 'qwen3.6:35b-a3b', 'model_provider': 'ollama'}, id='lc_run--019ecc6f-b506-75a2-81f4-1e9add64775b-0', tool_calls=[{'name': 'query_sql', 'args': {'query': "SELECT t.description FROM employees e JOIN profile p ON p.employees_id = e.id JOIN projects pr ON pr.profile_id = p.id JOIN tasks t ON t.projects_id = pr.id WHERE e.name = 'Tommy Parker'"}, 'id': 'c279cbad-ad68-4f34-bbfe-8c010ef869d5', 'type': 'tool_call'}], inva

{'firstTry': True,
 'isOk': True,
 'answer': '[["Up-sized 4thgeneration methodology"]]',
 'query': "SELECT t.description FROM employees e JOIN profile p ON p.employees_id = e.id JOIN projects pr ON pr.profile_id = p.id JOIN tasks t ON t.projects_id = pr.id WHERE e.name = 'Tommy Parker'",
 'modelAnswer': 'The task description for employee Tommy Parker is: **Up-sized 4thgeneration methodology**.'}

The answer from the tool call is stored as a JSON string in the `answer` property.

In the file `data/test_queries_v3.json`, you will find the reference query for this question: `SELECT COUNT(*) FROM employees`, which was written manually. Although the generated query is not identical, it produces the same result, so the comparison is straightforward: `answer == answer_ref`.

In other cases, the result is a JSON string containing a row set, represented as an array of arrays. To verify the query results, we created the `compare` function in the Python module `comparesql`.

For more details, see the `comparesql` module.

Our test cases are organized into four files containing JSON data:

- **test_queries_v3.json**: Basic test cases with questions of varying complexity.
- **test_queries_no.json**: None; questions that cannot be answered or yield no result.
- **test_queries_va.json**: Variant; variations of basic questions where the reference result remains the same.
- **test_queries_ex.json**: Extra; additional questions with relatively low complexity.

Each file contains an array of test cases. A test case is structured as follows:

```json
{
    "question": "Please give me the number of employees.",
    "js_query_ref_type": "js",
    "js_query_ref": "var result = data.employees.length;",
    "sql_query_ref": "SELECT COUNT(*) FROM employees",
    "type": "count"
}
```

- **question**: The question about the test data, expressed in natural language.
- **js_query_ref**: The reference JavaScript query to fetch the data.
- **sql_query_ref**: The reference SQL query to fetch the data.
- **type**: The type of query.

For the benchmark, we first use our agent to fetch the data based on the text in `question`. Then, we fetch the data using the SQL query `sql_query_ref`. Both results are compared using the function `compare`. If the comparison is successful, the test case is marked with `answerOK[i] = 1`; otherwise, it is marked with `answerOK[i] = 0`. The comparison results are finally saved in a pandas DataFrame.

In [5]:
import json
import pandas as pd
import comparesql
  
questions=[]
answerOK=[]
category=[]
with open("log_sql.txt", "w", encoding="utf-8") as f:
   f.write("Log\n")

def add_log(line):
   with open("log_sql.txt", "a", encoding="utf-8") as f:
      f.write(str(line)+"\n")

def add_error(cat,question, itry):
   if itry==2:
      category.append(cat)
      questions.append(question)
      answerOK.append(0)

def check_answer_sql(questions, answerOK, qa):
    q=qa["question"]
    sql=qa["sql_query_ref"]
    ref=query_sqlite3_db(sql)
    sref=json.dumps(ref)
    atype=qa["type"]
    add_log(f"Question: {q}")
    answer=get_answer(q)
    add_log(answer)
    add_log('###############')
    add_log(sref)
    iOKAnswer=0
    jref=json.loads(sref)
    janswer=[[]]
    sanswer=answer['answer']
    if len(sanswer):
      janswer=json.loads(answer['answer'])
    copo=False
    if atype=='no-tool-call':
      copo=len(sanswer)==0  
    else:
      copo=comparesql.compare(janswer,jref)
    if copo:
        iOKAnswer=1
    answerOK.append(iOKAnswer)
    questions.append(q)
    add_log("----")


We start with the category basic test cases.

In [6]:
import json
import time
with open('data/test_queries_v4.json') as f:
    testqueries = json.load(f) 
iQuestion=0    
for qa in testqueries['data'][0:25]:
     print(iQuestion)
     iQuestion += 1
     for ity in range(3):
        try:
          check_answer_sql(questions, answerOK, qa)
          category.append('yes')
          time.sleep(5)
          break
        except Exception as e:
          add_error('yes', qa['question'],  ity)
          add_log(f"Error checking answer: {e}")
          time.sleep(5)
    

0
{'messages': [HumanMessage(content='Please give me the number of employees.', additional_kwargs={}, response_metadata={}, id='7fcbfb4a-ed4d-4c95-a6b7-aa98c90cbcc5'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b-a3b', 'created_at': '2026-06-30T09:50:55.470996382Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16158347805, 'load_duration': 210089878, 'prompt_eval_count': 1023, 'prompt_eval_duration': 9700305000, 'eval_count': 124, 'eval_duration': 6235393000, 'logprobs': None, 'model_name': 'qwen3.6:35b-a3b', 'model_provider': 'ollama'}, id='lc_run--019f17f0-1ac0-7ce2-adf9-0a4f2bdee41d-0', tool_calls=[{'name': 'query_sql', 'args': {'query': 'SELECT COUNT(*) FROM employees'}, 'id': '125936d6-abe9-4628-8166-deceb2eb7461', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1023, 'output_tokens': 124, 'total_tokens': 1147}),
              ToolMessage(content='[[201]]', name='query_sql', id='0f110e1c-

None test cases.

In [8]:
import json
qs_safe=questions.copy()
aok_safe=answerOK.copy()
cat_safe=category.copy()

with open('data/test_queries_no.json') as f:
    testqueries = json.load(f) 
iQuestion=0
for qa in testqueries['data'][0:25]:
     print(iQuestion)
     iQuestion += 1
     for ity in range(3):
        try:
          check_answer_sql(questions, answerOK, qa)
          category.append('no')
          time.sleep(5)
          break
        except Exception as e:
          add_error('no', qa['question'],  ity)
          add_log(f"Error checking answer: {e}")
          time.sleep(5)

0
{'messages': [HumanMessage(content='Please give me the number of all clowns.', additional_kwargs={}, response_metadata={}, id='b3d1afe2-c8ab-4140-9151-73d7bd6cce57'),
              AIMessage(content='Based on the provided database schema, there is no table or data related to "clowns". Therefore, I cannot provide a count for them. If you meant to ask for the number of employees, tasks, projects, or another specific entity in the database, please clarify and I\'ll be happy to help!', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b-a3b', 'created_at': '2026-06-15T18:56:23.740943159Z', 'done': True, 'done_reason': 'stop', 'total_duration': 108496705196, 'load_duration': 213634475, 'prompt_eval_count': 1025, 'prompt_eval_duration': 9815018000, 'eval_count': 1887, 'eval_duration': 98457018000, 'logprobs': None, 'model_name': 'qwen3.6:35b-a3b', 'model_provider': 'ollama'}, id='lc_run--019ecca2-b2f3-75f2-9c30-b425c36c949f-0', tool_calls=[], invalid_tool_calls=[], usage_metadat

Variant test cases.

In [9]:
import json
qs_safe=questions.copy()
aok_safe=answerOK.copy()
cat_safe=category.copy()
with open('data/test_queries_va.json') as f:
    testqueries = json.load(f) 
for qa in testqueries['data'][0:25]:
     for ity in range(3):
        try:
          check_answer_sql(questions, answerOK, qa)
          category.append('va')
          time.sleep(5)
          break
        except Exception as e:
          add_error('no', qa['question'],  ity)
          add_log(f"Error checking answer: {e}")
          time.sleep(5)

{'messages': [HumanMessage(content='What is the total headcount of staff in the company?', additional_kwargs={}, response_metadata={}, id='ca2f57db-0a48-4b0b-aa9d-70be7e92058c'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b-a3b', 'created_at': '2026-06-15T20:15:25.5012467Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20332948890, 'load_duration': 218779985, 'prompt_eval_count': 1027, 'prompt_eval_duration': 9855000000, 'eval_count': 202, 'eval_duration': 10249024000, 'logprobs': None, 'model_name': 'qwen3.6:35b-a3b', 'model_provider': 'ollama'}, id='lc_run--019eccec-65c1-7da3-ac13-ac5910be0a73-0', tool_calls=[{'name': 'query_sql', 'args': {'query': 'SELECT COUNT(*) FROM employees'}, 'id': '2a32db89-1777-4a61-9a9b-da47bc462253', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1027, 'output_tokens': 202, 'total_tokens': 1229}),
              ToolMessage(content='[[201]]', name='query_sql', id=

Extra test cases.

In [10]:
qs=questions.copy()
aok=answerOK.copy()
cat=category.copy()

with open('data/test_queries_ex.json') as f:
    testqueries = json.load(f) 

test_data=testqueries['data']

for qa in testqueries['data'][0:25]:
     for ity in range(3):
        try:
          check_answer_sql(questions, answerOK, qa)
          category.append('ex')
          time.sleep(5)
          break
        except Exception as e:
          add_error('no', qa['question'],  ity)
          add_log(f"Error checking answer: {e}")
          time.sleep(5)


{'messages': [HumanMessage(content='Which employees (identified by id and name) have a name containing Haley?', additional_kwargs={}, response_metadata={}, id='1c296e5d-ade6-4bed-a98b-1cb93fe207ce'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b-a3b', 'created_at': '2026-06-15T21:08:48.812139976Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18522996626, 'load_duration': 214476518, 'prompt_eval_count': 1030, 'prompt_eval_duration': 9890212000, 'eval_count': 166, 'eval_duration': 8406638000, 'logprobs': None, 'model_name': 'qwen3.6:35b-a3b', 'model_provider': 'ollama'}, id='lc_run--019ecd1d-4dc2-7340-a4f1-9455937315d7-0', tool_calls=[{'name': 'query_sql', 'args': {'query': "SELECT id, name FROM employees WHERE name LIKE '%Haley%'"}, 'id': '8943861b-9a48-49e7-b2d1-53a4ad3d981f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1030, 'output_tokens': 166, 'total_tokens': 1196}),
              Tool

Now that we have the results in the `questions`, `answerOK`, and `category` lists, we can create a DataFrame and save it for future reference.

In [7]:
df = pd.DataFrame({
  "questions":questions,
  "okAnswer": answerOK  ,
  "category": category  
}                 
)
df.to_pickle("data/result_sql.pkl")


For a comparison with the corresponding SQL queries, see the notebook `compare_sql_js.ipynb`. Here, we will only provide the percentage of correct answers.

In [8]:
df['okAnswer'].sum()/len(df['okAnswer'])*100

np.float64(68.0)

In [13]:
df

,questions,okAnswer,category
0,Please give me the number of employees.,1,yes
1,Please give me the names of all employees.,1,yes
2,Please give me the email addresses of all empl...,1,yes
3,Please give me the addresses of employee Sara ...,1,yes
4,Please give me all project names.,1,yes
...,...,...,...
95,How many tasks are there in total?,1,ex
96,Wich is the oldest date when the certificate w...,1,ex
97,I need all differnt task descriptions.,1,ex
98,How many differnt secondary skills exists?,0,ex
